In [ ]:
import os
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import tensorly as tl
import dask.array as da
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import yeojohnson
import umap


In [ ]:
svd_dir = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_tensor_analysis/svd")

mode_names = ["Mode 0: Segments", "Mode 1: Samples (Rows)", "Mode 2: Samples (Cols)"]

variance_dfs = {}
for mode in range(3):
    variance_dfs[mode] = pd.read_parquet(svd_dir / f"variance_mode{mode}.parquet")
    S = np.load(svd_dir / f"S_mode{mode}.npy")
    print(f"{mode_names[mode]}: {len(S)} singular values")
    df = variance_dfs[mode]
    if (df["cumulative_variance"] >= 0.90).any():
        n90 = df.loc[df["cumulative_variance"] >= 0.90, "component"].iloc[0]
        print(f"  90% variance at component {n90}")
    if (df["cumulative_variance"] >= 0.95).any():
        n95 = df.loc[df["cumulative_variance"] >= 0.95, "component"].iloc[0]
        print(f"  95% variance at component {n95}")

In [ ]:
# --- Scree plots: 2 rows × 3 columns ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for mode in range(3):
    df = variance_dfs[mode]
    
    # For modes 1 & 2, limit to first 50 components for readability
    max_show = len(df) if mode == 0 else min(50, len(df))
    d = df.iloc[:max_show]
    
    # Top row: singular values
    axes[0, mode].plot(d["component"], d["singular_value"], "o-", markersize=4)
    axes[0, mode].set_xlabel("Component")
    axes[0, mode].set_ylabel("Singular Value")
    suffix = f" (first {max_show})" if mode > 0 else ""
    axes[0, mode].set_title(f"{mode_names[mode]} — Singular Values{suffix}")
    axes[0, mode].grid(True, alpha=0.3)
    
    # Bottom row: cumulative variance
    axes[1, mode].plot(d["component"], d["cumulative_variance"], "o-", markersize=4)
    axes[1, mode].axhline(y=0.90, color="r", linestyle="--", label="90%")
    axes[1, mode].axhline(y=0.95, color="orange", linestyle="--", label="95%")
    axes[1, mode].set_xlabel("Number of Components")
    axes[1, mode].set_ylabel("Cumulative Variance Explained")
    axes[1, mode].set_title(f"{mode_names[mode]} — Cumulative Variance{suffix}")
    axes[1, mode].legend()
    axes[1, mode].grid(True, alpha=0.3)

plt.suptitle("Scree Plots — SVD on Tensor Unfoldings", fontsize=16, y=1.00)
plt.tight_layout()
plt.show()